PASO 3  ·  Cargar el dataset al entorno Google Colab  

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Ajusta la ruta según donde guardaste el archivo en tu Drive
file_path = '/content/drive/MyDrive/salespredict/sales_5000000.csv'

# Es posible que necesites usar low_memory=False por el tamaño del archivo
df = pd.read_csv(file_path, low_memory=False)

print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

Mounted at /content/drive
El dataset tiene 5000000 filas y 14 columnas.


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Australia and Oceania,Palau,Office Supplies,Online,H,2020-03-06,517073523,2020-03-26,2401,651.21,524.96,1563555.21,1260428.96,303126.25
1,Europe,Poland,Beverages,Online,L,2014-04-18,380507028,2014-05-26,9340,47.45,31.79,443183.00,296918.60,146264.40
2,North America,Canada,Cereal,Online,M,2019-01-08,504055583,2019-01-31,103,205.70,117.11,21187.10,12062.33,9124.77
3,Europe,Belarus,Snacks,Online,C,2018-01-19,954955518,2018-02-27,1414,152.58,97.44,215748.12,137780.16,77967.96
4,Middle East and North Africa,Oman,Cereal,Offline,H,2023-04-26,970755660,2023-06-02,7027,205.70,117.11,1445453.90,822931.97,622521.93


PASO 4: PIPELINE ETL CON PYSPARK

In [7]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, month, year, dayofweek, to_date, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

# Initialize Spark Session
spark = SparkSession.builder.appName("SalesPredictETL").getOrCreate()

# Define file_path here for robustness
file_path = '/content/drive/MyDrive/salespredict/sales_5000000.csv'

# Leer el CSV, manejando la gran cantidad de columnas y posibles errores
sdf = spark.read.csv(file_path, header=True, inferSchema=True)

print(f"Filas leídas: {sdf.count()}")

# --- 1. Limpieza de Datos ---
# Eliminar filas con nulos en columnas críticas
# Se cambia 'Sales' por 'Total Revenue' porque 'Sales' no existe.
# Se elimina 'ProductLine' y 'CustomerName' de las columnas críticas
# ya que tampoco están presentes en el dataset cargado.
columnas_criticas = ['Total Revenue', 'Order Date']
sdf = sdf.dropna(subset=columnas_criticas)

# Limpiar columna 'Total Revenue' (puede tener '$' o comas si no fuera inferido como numérico)
# Aunque inferSchema=True debería manejar esto si el formato es consistente.
# La columna 'Total Revenue' ya es numérica, por lo que esta línea podría no ser estrictamente necesaria
# o requerir ajuste si hay valores no numéricos. Para este caso, se asume que inferSchema lo maneja.
sdf = sdf.withColumn('Total Revenue', regexp_replace(col('Total Revenue'), '[$,]', '').cast(DoubleType()))

# Asegurar que 'Units Sold' y 'Unit Price' son numéricos
sdf = sdf.withColumn('Units Sold', col('Units Sold').cast(IntegerType()))
sdf = sdf.withColumn('Unit Price', col('Unit Price').cast(DoubleType()))

# --- 2. Transformación de Tipos y Variables Derivadas ---
# Convertir 'Order Date' a tipo fecha. Ajusta el formato si es necesario.
# El formato 'M/d/yyyy' podría no ser el correcto, el dataframe de pandas muestra 'YYYY-MM-DD'.
# Se ajusta a 'yyyy-MM-dd'.
sdf = sdf.withColumn('OrderDate', to_date('Order Date', 'yyyy-MM-dd'))

# Crear variables derivadas
sdf = sdf.withColumn('year', year('OrderDate'))
sdf = sdf.withColumn('month', month('OrderDate'))
sdf = sdf.withColumn('day_of_week', dayofweek('OrderDate'))

# Crear una categoría de 'Tamaño de Venta' basada en 'Total Revenue'
sdf = sdf.withColumn('sales_category',
    when(col('Total Revenue') < 1000, 'Baja')
    .when(col('Total Revenue') < 5000, 'Media')
    .otherwise('Alta')
)

# --- 3. Guardar el resultado en formato Parquet ---
output_path = '/content/drive/MyDrive/salespredict/sales_clean'
sdf.write.mode('overwrite').parquet(output_path)

print("¡ETL Completado!")
sdf.printSchema()
# Se ajustan los nombres de las columnas para la muestra
sdf.select('Order ID', 'OrderDate', 'Item Type', 'Total Revenue', 'year', 'month', 'sales_category').show(5)

Filas leídas: 5000000
¡ETL Completado!
root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Item Type: string (nullable = true)
 |-- Sales Channel: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Order ID: integer (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Unit Cost: double (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Cost: double (nullable = true)
 |-- Total Profit: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- sales_category: string (nullable = false)

+---------+----------+---------------+-------------+----+-----+--------------+
| Order ID| OrderDate|      Item Type|Total Revenue|year|month|sales_category|
+-----

PASO 5: Modelo Predictivo (Regresión) con Scikit-learn

In [15]:
from pyspark.sql.functions import col
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import pyspark
from pyspark.sql import SparkSession
import os
import numpy as np

# Initialize Spark Session
try:
    spark
except NameError:
    spark = SparkSession.builder.appName("SalesPredictML").getOrCreate()

# Define output_path
output_path = '/content/drive/MyDrive/salespredict/sales_clean'

# Check if output_path exists
if not os.path.exists(output_path):
    print(f"Error: El directorio Parquet '{output_path}' no existe. Asegúrate de ejecutar el PASO 4 (ETL) primero.")
    raise FileNotFoundError(f"Parquet directory not found: {output_path}. Please run the ETL step (PASO 4) first.")

# Cargar los datos limpios desde Parquet (10% sample)
sdf_sample = spark.read.parquet(output_path).sample(0.1, seed=42)
df_pandas = sdf_sample.toPandas()

print(f"Tamaño de la muestra para el modelo: {df_pandas.shape}")

# --- PREPROCESAMIENTO CORREGIDO ---
# 1. DEFINIR FEATURES CORRECTOS (EXCLUYENDO UNITS SOLD Y UNIT PRICE)
# Solo usamos variables que estarían disponibles ANTES de la venta
features_cols = [
    'year',          # Año de la venta
    'month',         # Mes de la venta
    'day_of_week',   # Día de la semana
    'Item Type',     # Tipo de producto (categórica)
    'Country',       # País del cliente (si existe)
    'Territory',     # Territorio (si existe)
    'DealSize'       # Tamaño del trato (si existe)
]

# NOTA: Excluimos completamente 'Units Sold', 'Unit Price', 'sales_category'
# Porque 'sales_category' está basada en el monto total de venta (Sales/Total Revenue)
# y eso crearía fuga de datos

target_col = 'Total Revenue'  # Nuestro objetivo a predecir

# 2. Filtrar solo las columnas que realmente existen en tu dataset
# Verifica qué columnas tienes disponibles
columnas_disponibles = df_pandas.columns.tolist()
print(f"\nColumnas disponibles en el dataset: {columnas_disponibles}")

# Filtrar features_cols para incluir solo las que existen
features_cols_existentes = [col for col in features_cols if col in columnas_disponibles]
print(f"\nFeatures a utilizar: {features_cols_existentes}")

# 3. Crear el DataFrame para ML
df_ml = df_pandas[features_cols_existentes + [target_col]].dropna()

# 4. Limpiar valores infinitos
print(f"Filas antes de limpiar: {len(df_ml)}")
df_ml = df_ml.replace([np.inf, -np.inf], np.nan).dropna()
print(f"Filas después de limpiar: {len(df_ml)}")

# 5. Codificar variables categóricas
label_encoders = {}
categorical_cols = [col for col in features_cols_existentes if df_ml[col].dtype == 'object']
for col_name in categorical_cols:
    le = LabelEncoder()
    df_ml[col_name] = le.fit_transform(df_ml[col_name].astype(str))
    label_encoders[col_name] = le
    print(f"Codificada columna: {col_name}")

# 6. Separar features (X) y target (y)
X = df_ml[features_cols_existentes]
y = df_ml[target_col]

print(f"\nShape de X: {X.shape}")
print(f"Shape de y: {y.shape}")

# 7. Dividir en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 8. Entrenar Modelo Random Forest Regressor
print("\nEntrenando modelo...")
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_regressor.fit(X_train, y_train)

# 9. Evaluación del Modelo
y_pred = rf_regressor.predict(X_test)

print("\n--- Métricas de Evaluación (Modelo Corregido) ---")
print(f"R^2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error (MAE): ${mean_absolute_error(y_test, y_pred):.2f}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")

# 10. Importancia de las características
importances = rf_regressor.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\n--- Importancia de Características (Modelo Corregido) ---")
print(feature_importance_df)

# 11. Análisis de residuos
residuos = y_test - y_pred
print(f"\n--- Análisis de Residuos ---")
print(f"Media de residuos: ${residuos.mean():.2f}")
print(f"Desviación estándar de residuos: ${residuos.std():.2f}")
print(f"Error porcentual promedio: {(abs(residuos/y_test).mean() * 100):.2f}%")

# 12. Predicción de ejemplo
print("\n--- Ejemplo de Predicciones ---")
ejemplo_df = pd.DataFrame({
    'Real': y_test[:5].values,
    'Predicción': y_pred[:5]
})
print(ejemplo_df)

Tamaño de la muestra para el modelo: (499743, 19)

Columnas disponibles en el dataset: ['Region', 'Country', 'Item Type', 'Sales Channel', 'Order Priority', 'Order Date', 'Order ID', 'Ship Date', 'Units Sold', 'Unit Price', 'Unit Cost', 'Total Revenue', 'Total Cost', 'Total Profit', 'OrderDate', 'year', 'month', 'day_of_week', 'sales_category']

Features a utilizar: ['year', 'month', 'day_of_week', 'Item Type', 'Country']
Filas antes de limpiar: 499743
Filas después de limpiar: 499743
Codificada columna: Item Type
Codificada columna: Country

Shape de X: (499743, 5)
Shape de y: (499743,)

Entrenando modelo...

--- Métricas de Evaluación (Modelo Corregido) ---
R^2 Score: 0.5396
Mean Absolute Error (MAE): $666977.98
Root Mean Squared Error (RMSE): $994475.58

--- Importancia de Características (Modelo Corregido) ---
       Feature  Importance
3    Item Type    0.986540
4      Country    0.005680
0         year    0.002777
1        month    0.002719
2  day_of_week    0.002285

--- Análisi

In [20]:
# --- Exportar datos para Neo4j ---
from pyspark.sql.functions import col
import os # Import the os module for directory operations

# Cargar los datos limpios
sdf_clean = spark.read.parquet('/content/drive/MyDrive/salespredict/sales_clean') # Corrected path

# Tomar una muestra representativa (ej. 50,000 registros para no sobrecargar Neo4j)
sdf_sample = sdf_clean.sample(0.01, seed=42)  # 1% de los datos

# Seleccionar solo las columnas necesarias para el grafo
# Corrected column names based on available schema after ETL (PASO 4)
sdf_graph = sdf_sample.select(
    'Order ID',
    'Item Type', # Replaced 'ProductLine'
    'Total Revenue', # Replaced 'Sales'
    'Region',
    'Country',
    'Sales Channel',
    'Units Sold'
)

# Convertir a Pandas y guardar como CSV
df_graph = sdf_graph.toPandas()

# Guardar en Google Drive
output_csv_path = '/content/drive/MyDrive/salespredict/data/graph_data_sample.csv'

# --- Create the directory if it does not exist ---
output_dir = os.path.dirname(output_csv_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir) # This creates the directory and any necessary parent directories

df_graph.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"✅ Datos para Neo4j guardados en: {output_csv_path}")
print(f"Total de registros: {len(df_graph)}")
print("\nMuestra de los datos:")
df_graph.head()

✅ Datos para Neo4j guardados en: /content/drive/MyDrive/salespredict/data/graph_data_sample.csv
Total de registros: 49731

Muestra de los datos:


,Order ID,Item Type,Total Revenue,Region,Country,Sales Channel,Units Sold
0,852520883,Cosmetics,3816318.80,Sub-Saharan Africa,South Africa,Online,8729
1,523031461,Beverages,260358.15,Asia,Indonesia,Online,5487
2,649103868,Clothes,340297.92,Sub-Saharan Africa,Rwanda,Offline,3114
3,800648248,Vegetables,1303963.84,Sub-Saharan Africa,Chad,Offline,8464
4,466824805,Baby Food,686447.92,Europe,Albania,Offline,2689


In [22]:
# En Colab, ejecuta esto para ver el contenido
with open('/content/drive/MyDrive/salespredict/data/graph_data_sample.csv', 'r') as f:
    print(f.read()[:2000])  # Muestra las primeras líneas

Order ID,Item Type,Total Revenue,Region,Country,Sales Channel,Units Sold
852520883,Cosmetics,3816318.8,Sub-Saharan Africa,South Africa,Online,8729
523031461,Beverages,260358.15,Asia,Indonesia,Online,5487
649103868,Clothes,340297.92,Sub-Saharan Africa,Rwanda,Offline,3114
800648248,Vegetables,1303963.84,Sub-Saharan Africa,Chad,Offline,8464
466824805,Baby Food,686447.92,Europe,Albania,Offline,2689
659945142,Personal Care,105268.24,Central America and the Caribbean,Grenada,Offline,1288
956425368,Vegetables,982132.5,Europe,Cyprus,Online,6375
765241801,Cereal,186981.3,Europe,Latvia,Online,909
947203719,Cosmetics,3272004.8,Central America and the Caribbean,Saint Vincent and the Grenadines,Online,7484
170234286,Household,3316624.01,Europe,Belgium,Offline,4963
529937827,Cosmetics,4037104.8,Central America and the Caribbean,Dominica,Online,9234
287466990,Snacks,1026100.5,Europe,Albania,Online,6725
673078525,Baby Food,1977909.44,Australia and Oceania,New Zealand,Offline,7748
642523229,Snacks,8782